# Does the gingerbread season start earlier every year? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/gingerbread-earlier-every-year/story_004_gingerbread.ipynb)

This R notebook recomputes every statistic in
[Does the gingerbread season start earlier every year?](https://pointzerofive.net/2026/10/gingerbread-earlier-every-year),
story #004 on *pointzerofive*, from the Google Trends file. Colab opens it with the R runtime; run it
top to bottom (`Runtime → Run all`). Nothing here is random, so every number matches exactly.

In [ ]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/gingerbread-earlier-every-year/"
REPO <- ".."

# A file of this story: next to the notebook (pointzerofive-data), in the website's repository,
# else downloaded from pointzerofive-data.
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}

trends <- read.csv(fetch_path("notebooks/raw/gingerbread/trends_daily.csv"), encoding = "UTF-8",
                   stringsAsFactors = FALSE)
published <- fromJSON(fetch_path("data/gingerbread-earlier-every-year.json"), simplifyVector = FALSE)
cat(nrow(trends), "daily values\n")

## 1. The data

Google Trends, web search, Germany. One request per season, 1 June to 31 December, with the terms
"Lebkuchen", "Spekulatius" and "Dominosteine" together: 22 seasons × 214 days × 3 terms. Within a
season the three series share one scale from 0 to 100; between seasons the scales differ, which
does not matter because every season is compared only with its own peak. Google rounds small
volumes to 0, which hits the early years hardest.

In [ ]:
SEASONS <- 2004:2025
DAYS <- 214L
TERMS <- c(lebkuchen = "Lebkuchen", spekulatius = "Spekulatius", dominosteine = "Dominosteine")
stopifnot(nrow(trends) == length(SEASONS) * DAYS * length(TERMS), all(trends$partial == 0))
trends$day <- as.integer(as.Date(trends$date) - as.Date(sprintf("%d-06-01", trends$season)))
stopifnot(all(trends$day >= 0 & trends$day < DAYS))

lk <- trends[trends$term == "Lebkuchen", ]
zeros <- tapply(lk$value == 0, lk$season, sum)
print(zeros)
for (s in published$seasons) stopifnot(zeros[[as.character(s$season)]] == s$zeros)

## 2. The start of each season

Seven-day moving mean, ending on each day. The peak is the largest mean of the season. The start
(onset) is the first day from which the mean stays at or above a share of the peak until the peak,
counted in days after 1 June.

In [ ]:
# Daily values per season (rows) and day after 1 June (columns) for one treat.
series <- function(treat) {
  terms <- if (treat == "all") unname(TERMS) else TERMS[[treat]]
  d <- trends[trends$term %in% terms, ]
  tapply(d$value, list(d$season, d$day), sum)
}
moving_mean <- function(x, k = 7L) as.numeric(stats::filter(x, rep(1 / k, k), sides = 1))
onset <- function(x, share) {
  m <- moving_mean(x)
  peak <- which.max(m)
  level <- share * m[peak]
  d <- peak
  while (d > 7L && m[d - 1] >= level) d <- d - 1
  as.integer(d - 1L)
}

onsets <- list()
for (treat in c("lebkuchen", "spekulatius", "all")) {
  s <- series(treat)
  for (th in c("10", "20", "30")) {
    onsets[[paste(treat, th, sep = "_")]] <-
      vapply(rownames(s), function(y) onset(s[y, ], as.numeric(th) / 100), 0L)
  }
}
for (i in seq_along(SEASONS)) {
  p <- published$seasons[[i]]
  stopifnot(p$season == SEASONS[i])
  for (k in names(onsets)) stopifnot(onsets[[k]][[i]] == p[[k]])
}
s <- series("lebkuchen")
peak_day <- apply(s, 1, function(x) which.max(moving_mean(x)) - 1L)
for (i in seq_along(SEASONS)) stopifnot(peak_day[[i]] == published$seasons[[i]]$peakDay)
date_of <- function(d) format(as.Date("2001-06-01") + d, "%d %b")
data.frame(season = SEASONS, onset = date_of(onsets$lebkuchen_20), peak = date_of(peak_day),
           row.names = NULL)

## 3. Fig. 1: the search curves

The chart draws the seven-day means of "Lebkuchen" from 7 June on, each season scaled to its own
peak; black is the mean of the first five seasons, red the mean of the last five.

In [ ]:
for (i in seq_along(SEASONS)) {
  mine <- round(moving_mean(s[i, ])[7:DAYS], 2)
  stopifnot(max(abs(mine - unlist(published$daily[[i]]$mean7))) < 1e-9)
}
scaled <- t(apply(s, 1, function(x) { m <- moving_mean(x)[7:DAYS]; 100 * m / max(m) }))
first5 <- colMeans(scaled[1:5, ]); last5 <- colMeans(scaled[18:22, ])
day <- 6:(DAYS - 1)
matplot(day, t(scaled), type = "l", lty = 1, col = "grey80", xlab = "days after 1 June",
        ylab = "search interest (% of peak)")
lines(day, first5, lwd = 2); lines(day, last5, lwd = 2, col = "#c0392b"); abline(h = 20, lty = 2)

## 4. All 27 calculation paths

Effect = minus the slope of the least-squares line of onset on year, in days earlier per year. The
p-value and the 95% confidence interval come from the t distribution with n − 2 degrees of
freedom.

In [ ]:
test_path <- function(years, o) {
  fit <- lm(o ~ years)
  co <- summary(fit)$coefficients
  b <- co["years", "Estimate"]; se <- co["years", "Std. Error"]; df <- fit$df.residual
  q <- qt(0.975, df)
  list(effect = -b, ci_low = -b - q * se, ci_high = -b + q * se, p = 2 * pt(-abs(b / se), df),
       n = length(years))
}
first_year <- c(all = 2004L, from2011 = 2011L, from2016 = 2016L)
rows <- list()
for (treat in c("lebkuchen", "spekulatius", "all")) for (th in c("10", "20", "30"))
  for (yrs in names(first_year)) {
    keep <- SEASONS >= first_year[[yrs]]
    r <- test_path(SEASONS[keep], onsets[[paste(treat, th, sep = "_")]][keep])
    rows[[length(rows) + 1]] <- data.frame(treat = treat, threshold = th, years = yrs, r)
  }
specs <- do.call(rbind, rows)
for (i in seq_len(nrow(specs))) {
  p <- published$specs[[i]]
  stopifnot(identical(unname(unlist(p$choices)), unname(unlist(specs[i, 1:3]))), p$n == specs$n[i])
  for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(p[[k]] - specs[[k]][i]) < 1e-9)
}
print(specs, digits = 3)

## 5. The verdict and the result table

The main calculation was fixed in the pitch before the download: Lebkuchen, 20% of the peak, all
seasons. "Yes" needs p < 0.05 and a positive effect (earlier). Smallest relevant effect: 0.5 days
per year.

In [ ]:
main <- specs[specs$treat == "lebkuchen" & specs$threshold == "20" & specs$years == "all", ]
stopifnot(published$claim$smallest_relevant_effect == 0.5)
verdict <- if (main$p < 0.05 && main$effect > 0) "yes" else "no"
cat(sprintf("main: %+.3f days earlier per year, 95%% CI [%+.3f, %+.3f], p = %.4f, n = %d -> %s\n",
            main$effect, main$ci_low, main$ci_high, main$p, main$n, verdict))
stopifnot(verdict == "yes")
yes <- specs$p < 0.05 & specs$effect > 0
cat(sprintf("yes on %d of %d paths, %d of them Lebkuchen\n", sum(yes), nrow(specs),
            sum(yes & specs$treat == "lebkuchen")))

# The result table under "The hypothesis test": the regression on years since 2004.
o <- onsets$lebkuchen_20
x <- SEASONS - 2004L
fit <- lm(o ~ x); sm <- summary(fit); co <- sm$coefficients; df <- fit$df.residual
row_t <- function(est, se) c(est, se, est - qt(0.975, df) * se, est + qt(0.975, df) * se,
                             2 * pt(-abs(est / se), df))
mine <- rbind(row_t(co[1, 1], co[1, 2]), row_t(-co[2, 1], co[2, 2]))
table_rows <- do.call(rbind, lapply(published$estimation$rows, function(r) unlist(r$values)))
stopifnot(max(abs(mine - table_rows)) < 1e-9)
r <- residuals(fit)
stats <- c(sm$r.squared, sm$adj.r.squared, sm$sigma, sum(diff(r)^2) / sum(r^2))
pub_stats <- vapply(published$estimation$stats$left, function(s) s$value, 0)
stopifnot(max(abs(stats - pub_stats)) < 1e-9)
dimnames(mine) <- list(c("Onset in 2004 (C)", "Days earlier per year"),
                       c("Coefficient", "Std. Error", "Lower 95%", "Upper 95%", "Prob."))
print(round(mine, 6))

# The fitted line of Fig. 2 and the check without 2004 (done after the result, not in the pitch)
line <- coef(lm(o ~ SEASONS))
stopifnot(abs(line[[1]] - published$line$intercept) < 1e-6, abs(line[[2]] - published$line$slope) < 1e-9)
w <- test_path(SEASONS[-1], o[-1])
for (k in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(w[[k]] - published$checks$without2004[[k]]) < 1e-9)
cat(sprintf("without 2004: %+.3f [%+.3f, %+.3f], p = %.4f\n", w$effect, w$ci_low, w$ci_high, w$p))

## Data and methods

- **Source:** Google Trends, web search, Germany, downloaded on 16 October 2026 with
  `gingerbread_download.py`. "Source: Google Trends".
- **Pipeline:** `gingerbread_pipeline.R` (base R and `jsonlite`), console output in
  `gingerbread-earlier-every-year.output.txt`.
- **Pitch:** every choice was fixed in `pitch.md` before the download.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.

In [ ]:
cat("All statistics reproduced ✓\n")